In [27]:
import warnings
warnings.filterwarnings("ignore", message="The default value of `allowed_objects`")

In [28]:
from dotenv import load_dotenv

load_dotenv()

True

In [29]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

## Write to state

In [30]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [35]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(
    model="nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
    temperature=0.8
)

agent = create_agent(
    model=model,
    tools=[update_favourite_colour],
    #checkpointer=InMemorySaver(),
    state_schema=CustomState,
    system_prompt="You are a helpful assistant that helps the user keep track of their favourite colour. If the user reveals their favourite colour, update it in the state using the provided tool. If you dont know the user's favourite colour, ask them for it. Always respond in a friendly and helpful manner."
)

In [36]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is red")]},
    {"configurable": {"thread_id": "1"}}
)

KeyboardInterrupt: 

In [33]:
from pprint import pprint

pprint(response)
print(response["messages"][-1].content)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='c3c43229-ca5d-4a33-968a-679b8270c14c'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user just told me their favorite color is green. Let me check the tools available. There\'s a function called update_favourite_colour that takes a favourite_colour parameter. Since they provided the color, I should call that function to update the state. I need to make sure the argument is correctly formatted as a JSON object. The required parameter is favourite_colour, and the value here is "green". So I\'ll generate the tool call with that information. No need to ask for more details since the user already revealed their favorite color. Just execute the function call properly.\n', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'Okay, the user just told me their favorite colo

In [ ]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you? what is my favorite color?")],
    },
    {"configurable": {"thread_id": "1"}}
)

pprint(response)
print(response["messages"][-1].content)

## Read state

In [45]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    model=model,
    tools=[update_favourite_colour, read_favourite_colour],
    #checkpointer=InMemorySaver(),
    state_schema=CustomState,
    system_prompt="You are a helpful assistant that helps the user keep track of their favourite colour. If the user reveals their favourite colour, update it in the state using the provided tool. If not, fetch the user's favorite color from the state using the provided tool. If user's favourite colour is not in state, ask them for it. Always respond in a friendly and helpful manner."
)

In [46]:
response = agent.invoke(
    { "messages": [HumanMessage(content="Hi. what is my favorite colour?")]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)
print(response["messages"][-1].content)

{'messages': [HumanMessage(content='Hi. what is my favorite colour?', additional_kwargs={}, response_metadata={}, id='aa016fd1-0ef0-4e2a-b6a4-e7ff2c966209'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user is asking, "Hi. what is my favorite colour?" So first, I need to figure out how to respond.\n\nLooking at the tools available, there\'s a function called read_favourite_colour that retrieves the user\'s favorite color from the state. Since the user is asking for their favorite color, I should use that tool. \n\nWait, the instructions say if the user reveals their favorite color, update it. But here, they\'re asking for it, not revealing it. So first step is to read the current favorite color. \n\nI need to call read_favourite_colour. No parameters needed for that function. So I\'ll generate the tool call for read_favourite_colour. Once I get the response, if there\'s a color, I can tell them. If not, then I need to ask them what

In [47]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is purple")]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)
print(response["messages"][-1].content)

{'favourite_colour': 'purple',
 'messages': [HumanMessage(content='My favourite colour is purple', additional_kwargs={}, response_metadata={}, id='12bd21c7-432c-4d45-9889-746bc861be5b'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user just said their favorite color is purple. Let me check the tools available. There\'s the update_favourite_colour function which takes a colour parameter. Since they revealed their favorite color, I should use that tool to update the state. The function requires the favourite_colour as a string. So I\'ll call update_favourite_colour with "purple" as the argument. No need to ask for more info here because they already provided it. Let me make sure the JSON is correctly formatted.\n', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'Okay, the user just said their favorite color is purple. Let me check the tools available. There\'s the update_favourite_colour function which 

In [48]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)
print(response["messages"][-1].content)

{'messages': [HumanMessage(content="What's my favourite colour?", additional_kwargs={}, response_metadata={}, id='ab4cd86d-4559-4839-8255-97a21d7739ae'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking, "What\'s my favourite colour?" Let me see how to handle this.\n\nFirst, I need to check if I have the user\'s favorite color already stored. The tools available are update_favourite_colour and read_favourite_colour. Since the user is asking for their favorite color, the appropriate tool here is read_favourite_colour. \n\nI should call read_favourite_colour to get the current favorite color from the state. If the state already has it, I can return that. If not, then I need to ask the user to provide it. \n\nWait, the instructions say that if the favorite color isn\'t in the state, I should ask them for it. So first step is to read the state. Let me make sure I use the correct function. The read_favourite_colour doesn\'t require any pa